# Building the Dataframe

This notebook builds the dataframe the XGBoost model reads (`model_dataframe_2023_Q1_to_2026_Q3.txt`): one row per 10-K or 10-Q, the 8 variables on a 12-month basis, and the bankruptcy label (1 if the company filed a bankruptcy 8-K, Item 1.03, within 365 days after the filing).

It needs the SEC Financial Statement Data Sets from 2023 Q1 to 2026 Q3 in `data/`, one folder per quarter (for example `data/2023Q1/sub.txt` and `data/2023Q1/num.txt`), from https://www.sec.gov/data-research/sec-markets-data/financial-statement-data-sets

In [1]:
import time

import numpy as np
import pandas as pd
import requests

QUARTERS = ["2023Q1", "2023Q2", "2023Q3", "2023Q4", "2024Q1", "2024Q2", "2024Q3", "2024Q4",
            "2025Q1", "2025Q2", "2025Q3", "2025Q4", "2026Q1", "2026Q2", "2026Q3"]

#the date the labels were made. A filing's answer is known once it is at least 365 days older than this.
#It's fixed so the dataframe comes out the same every time the notebook is run
LABEL_DATE = pd.Timestamp("2026-10-17")

#SEC asks for a name and email in the user agent
HEADERS = {"User-Agent": "Pavanchytanya Dharmapuri dharmapuri.p@northeastern.edu"}

## One Row per Filing

In [2]:
#the num files are too big to keep whole, so I only keep the tags we use as soon as each one is read
my_tags = ["Assets", "AssetsCurrent", "LiabilitiesCurrent", "Liabilities",
           "RevenueFromContractWithCustomerExcludingAssessedTax", "Revenues",
           "NetIncomeLoss", "OperatingIncomeLoss",
           "NetCashProvidedByUsedInOperatingActivities",
           "NetCashProvidedByUsedInOperatingActivitiesContinuingOperations"]

sub_list, num_list = [], []
for folder in reversed(QUARTERS):
    sub_list.append(pd.read_csv(f"data/{folder}/sub.txt", sep="\t", header=0, low_memory=False))
    num_q = pd.read_csv(f"data/{folder}/num.txt", sep="\t", header=0, low_memory=False)
    num_list.append(num_q[num_q["tag"].isin(my_tags)])
sub_concatenated = pd.concat(sub_list)
num_concatenated = pd.concat(num_list)
print("filings in sub:", len(sub_concatenated))

filings in sub: 102119


In [3]:
#only the company level totals, so I drop the rows where segments or coreg is filled in (those are
#breakdowns by business unit or sub-company). Then one row per filing and date that has any of our tags
numbers = num_concatenated[num_concatenated["segments"].isna() & num_concatenated["coreg"].isna()]
numbers = numbers.merge(sub_concatenated[["adsh", "cik", "name"]], on="adsh")
wide = numbers.pivot_table(index=["cik", "adsh", "name", "ddate"], columns="tag",
                           values="value", aggfunc="first").reset_index()
filings = wide[["cik", "adsh", "name", "ddate"]].sort_values(["cik", "ddate"]).reset_index(drop=True)

#each filing also repeats the numbers of older periods to compare with. sub has a 'period' column which is
#the filing's own balance sheet date, so I keep the row where ddate matches it
filings = filings.merge(sub_concatenated[["adsh", "period"]], on="adsh")
filings = filings[filings["ddate"] == filings["period"]].drop(columns="period").reset_index(drop=True)
print("rows (one per filing):", len(filings))

rows (one per filing): 101001


## 10-K and 10-Q Only

In [4]:
#I keep only 10-K and 10-Q and their amendments. Other forms like S-1 repeat a period that is already in a
#10-K or 10-Q, and 20-F, 40-F and 6-K are foreign companies, which don't file 8-Ks, so they would always be 0
model_df = filings.merge(sub_concatenated[["adsh", "form", "fp", "sic", "filed"]].drop_duplicates("adsh"), on="adsh")
print("rows before:", len(model_df))
print(model_df["form"].value_counts().head(10))
model_df = model_df[model_df["form"].isin(["10-K", "10-Q", "10-K/A", "10-Q/A"])]
print("rows with 10-K, 10-Q and amendments:", len(model_df))

#an amendment repeats the same period, so for each company and date I keep the one filed first,
#since that's the one we would have had at that time
model_df = model_df.sort_values(["cik", "ddate", "filed", "adsh"]).drop_duplicates(["cik", "ddate"]).reset_index(drop=True)
print("rows with one filing per company and date:", len(model_df))
print(model_df["form"].value_counts())

rows before: 101001
form
10-Q      63887
10-K      22650
20-F       3926
6-K        2252
S-1/A      1475
10-Q/A     1107
S-1        1087
10-K/A     1030
S-4/A       725
POS AM      617
Name: count, dtype: int64
rows with 10-K, 10-Q and amendments: 88674
rows with one filing per company and date: 86927
form
10-Q      63874
10-K      22643
10-Q/A      284
10-K/A      126
Name: count, dtype: int64


## The 8 Variables

In [5]:
#Balance sheet tags use qtrs 0. The other tags use the year so far: 1 for Q1, 2 for Q2, 3 for Q3 and 4 for a 10-K.
#If the first tag is empty I use the next one in the list
balance_tags = {
    "total_assets": ["Assets"],
    "current_assets": ["AssetsCurrent"],
    "current_liabilities": ["LiabilitiesCurrent"],
    "total_liabilities": ["Liabilities"],
    #only used to work out total liabilities when it's empty
    "liabilities_and_equity": ["LiabilitiesAndStockholdersEquity"],
    "equity_with_minority": ["StockholdersEquityIncludingPortionAttributableToNoncontrollingInterest", "StockholdersEquity"],
}
flow_tags = {
    "revenue": ["RevenueFromContractWithCustomerExcludingAssessedTax", "Revenues",
                "RevenueFromContractWithCustomerIncludingAssessedTax", "SalesRevenueNet"],
    "net_income": ["NetIncomeLoss", "ProfitLoss"],
    "operating_income": ["OperatingIncomeLoss"],
    "operating_cash_flow": ["NetCashProvidedByUsedInOperatingActivities",
                            "NetCashProvidedByUsedInOperatingActivitiesContinuingOperations"],
}
all_tags = sorted(set(sum(balance_tags.values(), []) + sum(flow_tags.values(), [])))
balance_tag_names = set(sum(balance_tags.values(), []))

#how many quarters the year so far covers
model_df["qtrs_wanted"] = model_df["fp"].map({"Q1": 1, "Q2": 2, "Q3": 3, "FY": 4})
model_df.loc[model_df["form"].str.startswith("10-K"), "qtrs_wanted"] = 4

#reading the num files again, this time with the qtrs and uom columns and the backup tags.
#Only company level totals in US dollars
values_list = []
for folder in QUARTERS:
    num_q = pd.read_csv(f"data/{folder}/num.txt", sep="\t", header=0, low_memory=False,
                        usecols=["adsh", "tag", "ddate", "qtrs", "uom", "segments", "coreg", "value"])
    num_q = num_q[num_q["tag"].isin(all_tags) & num_q["segments"].isna() & num_q["coreg"].isna()
                  & (num_q["uom"] == "USD") & num_q["value"].notna()]
    num_q = num_q.merge(model_df[["adsh", "ddate", "qtrs_wanted"]], on=["adsh", "ddate"])
    is_balance = num_q["tag"].isin(balance_tag_names)
    num_q = num_q[(is_balance & (num_q["qtrs"] == 0)) | (~is_balance & (num_q["qtrs"] == num_q["qtrs_wanted"]))]
    values_list.append(num_q[["adsh", "tag", "value"]])

values_long = pd.concat(values_list).drop_duplicates(["adsh", "tag"])
values_wide = values_long.pivot(index="adsh", columns="tag", values="value")
for tag in all_tags:
    if tag not in values_wide:
        values_wide[tag] = np.nan

#first tag in the list, then the next one where it's empty
for name, tag_list in {**balance_tags, **flow_tags}.items():
    column = values_wide[tag_list[0]]
    for backup in tag_list[1:]:
        column = column.fillna(values_wide[backup])
    values_wide[name] = column

#a lot of companies don't use the Liabilities tag, so total liabilities = liabilities and equity - equity
values_wide["total_liabilities"] = values_wide["total_liabilities"].fillna(
    values_wide["liabilities_and_equity"] - values_wide["equity_with_minority"])

variables = ["total_assets", "current_assets", "current_liabilities", "total_liabilities",
             "revenue", "net_income", "operating_income", "operating_cash_flow"]
model_df = model_df.merge(values_wide[variables], left_on="adsh", right_index=True, how="left")

#a 10-Q for Q2 has 6 months of revenue and a 10-K has 12, so I turn all of them into a 12 month number
#(times 4, 2 or 4/3) so that 10-Q and 10-K rows can be compared
for name in flow_tags:
    model_df[name] = model_df[name] * 4 / model_df["qtrs_wanted"]
print("share of rows with a value:")
print(model_df[variables].notna().mean().round(3))

share of rows with a value:
total_assets           0.992
current_assets         0.779
current_liabilities    0.776
total_liabilities      0.991
revenue                0.735
net_income             0.983
operating_income       0.759
operating_cash_flow    0.979
dtype: float64


## Bankruptcy Labels

In [6]:
#I ask EDGAR full text search once per year for every 8-K that has Item 1.03 (bankruptcy) in it,
#then match them to our filings by cik. 2021 and 2022 are only there to know which companies were
#already bankrupt in early 2023
def get_bankruptcy_8ks(start_date, end_date):
    results = []
    position = 0
    while True:
        params = {"forms": "8-K", "q": '"Item 1.03"', "startdt": start_date, "enddt": end_date, "from": position}
        hits = requests.get("https://efts.sec.gov/LATEST/search-index", params=params,
                            headers=HEADERS, timeout=60).json()["hits"]
        results += hits["hits"]
        position += len(hits["hits"])
        if len(hits["hits"]) == 0 or position >= hits["total"]["value"]:
            return results
        time.sleep(0.15)


event_rows = []
for year in range(2021, 2027):
    for hit in get_bankruptcy_8ks(f"{year}-01-01", f"{year}-12-31"):
        source = hit["_source"]
        #the search also finds 8-Ks that only mention Item 1.03, so I keep the ones where 1.03 is one of the items
        if "1.03" not in (source.get("items") or []):
            continue
        accession, document = hit["_id"].split(":")
        #some 8-Ks are filed for a parent company and its subsidiaries together, so they have more than one cik
        for cik in source["ciks"]:
            event_rows.append([int(cik), accession, source["file_date"]])

all_8ks = pd.DataFrame(event_rows, columns=["cik", "bk_adsh", "bk_date"]).drop_duplicates(["cik", "bk_adsh"])
all_8ks["bk_date"] = pd.to_datetime(all_8ks["bk_date"])
#only the 8-Ks we had when the labels were made, so a later run gives the same dataframe
all_8ks = all_8ks[all_8ks["bk_date"] <= LABEL_DATE]
print("bankruptcy 8-Ks 2021 to 2026:", all_8ks["bk_adsh"].nunique(), " companies:", all_8ks["cik"].nunique())

bankruptcy 8-Ks 2021 to 2026: 565  companies: 323


In [7]:
model_df["filed_date"] = pd.to_datetime(model_df["filed"].astype(int).astype(str), format="%Y%m%d")
pairs = model_df[["cik", "adsh", "filed_date"]].merge(all_8ks[["cik", "bk_date"]], on="cik")
pairs["days"] = (pairs["bk_date"] - pairs["filed_date"]).dt.days

#Bankruptcy = 1 if the company has a bankruptcy 8-K in the 365 days after the filing
next_8k = pairs[(pairs["days"] > 0) & (pairs["days"] <= 365)].groupby("adsh")["bk_date"].min()
model_df["bk_date"] = model_df["adsh"].map(next_8k)
model_df["days_to_bankruptcy"] = (model_df["bk_date"] - model_df["filed_date"]).dt.days
model_df["Bankruptcy"] = model_df["bk_date"].notna().astype(float)

#already bankrupt = there was a bankruptcy 8-K on the same day or in the 2 years before the filing.
#Those filings already say the company is bankrupt, so they are not a prediction
already = pairs[(pairs["days"] <= 0) & (pairs["days"] >= -730)]["adsh"].unique()
model_df["already_bankrupt"] = model_df["adsh"].isin(already)

#we can only know the answer for filings that are at least 365 days old, so every newer row is NaN (1s too)
model_df["label_known"] = model_df["filed_date"] + pd.Timedelta(days=365) <= LABEL_DATE
model_df.loc[~model_df["label_known"], "Bankruptcy"] = np.nan

model_df = model_df.sort_values(["cik", "ddate"]).reset_index(drop=True)
print(pd.crosstab(model_df["already_bankrupt"], model_df["Bankruptcy"].fillna(-1)).rename(columns={-1.0: "NaN"}))

Bankruptcy          NaN    0.0  1.0
already_bankrupt                   
False             22138  63928  609
True                 47    146   59


## The Rows the Model Trains On

In [8]:
#rows we can train on: label known and the company wasn't already bankrupt
train_ready = model_df[model_df["label_known"] & ~model_df["already_bankrupt"]]
print("all rows:", len(model_df), " rows we can train on:", len(train_ready))
print(train_ready["Bankruptcy"].value_counts())
print("companies:", train_ready["cik"].nunique(), " companies with a 1:", train_ready.loc[train_ready["Bankruptcy"] == 1, "cik"].nunique())
print("share of rows that are 1:", round(train_ready["Bankruptcy"].mean() * 100, 2), "%")

all rows: 86927  rows we can train on: 64537
Bankruptcy
0.0    63928
1.0      609
Name: count, dtype: int64
companies: 7137  companies with a 1: 204
share of rows that are 1: 0.94 %


In [9]:
#how many values are empty in each column, for the rows we can train on (in %).
#Current assets and current liabilities are empty mostly for banks, insurance companies and REITs, they don't
#split their balance sheet into current and non current. Revenue is empty mostly for companies with no sales yet
#(biotech, shell companies). XGBoost can work with empty values, so I leave them empty and don't fill them with 0 or the mean
print((train_ready[variables].isna().mean() * 100).round(1))
industry_groups = np.where(train_ready["sic"].between(6000, 6799), "finance, insurance, real estate", "other industries")
print((train_ready[["current_assets", "revenue", "operating_income"]].isna().groupby(industry_groups).mean() * 100).round(1))

total_assets            0.9
current_assets         21.8
current_liabilities    22.1
total_liabilities       0.9
revenue                26.1
net_income              1.6
operating_income       23.7
operating_cash_flow     1.8
dtype: float64
                                 current_assets  revenue  operating_income
finance, insurance, real estate            70.6     52.5              62.1
other industries                            6.1     17.7              11.3


In [10]:
model_df.to_csv("model_dataframe_2023_Q1_to_2026_Q3.txt", sep="\t", index=False)
model_df.head()

,cik,adsh,name,ddate,form,fp,sic,filed,qtrs_wanted,total_assets,...,revenue,net_income,operating_income,operating_cash_flow,filed_date,bk_date,days_to_bankruptcy,Bankruptcy,already_bankrupt,label_known
0,1750,0001104659-23-035153,AAR CORP,20230228,10-Q,Q3,3720.0,20230321,3.0,1.673300e+09,...,1.916267e+09,8.933333e+07,NaN,-2.920000e+07,2023-03-21,NaT,NaN,0.0,False,True
1,1750,0001104659-23-082069,AAR CORP,20230531,10-K,FY,3720.0,20230718,4.0,1.833100e+09,...,1.990500e+09,9.020000e+07,NaN,2.330000e+07,2023-07-18,NaT,NaN,0.0,False,True
2,1750,0001104659-23-103944,AAR CORP,20230831,10-Q,Q1,3720.0,20230926,1.0,1.954400e+09,...,2.198800e+09,-2.400000e+06,NaN,-7.480000e+07,2023-09-26,NaT,NaN,0.0,False,True
3,1750,0001104659-23-128321,AAR CORP,20231130,10-Q,Q2,3720.0,20231221,2.0,1.965600e+09,...,2.190200e+09,4.640000e+07,NaN,-2.600000e+06,2023-12-21,NaT,NaN,0.0,False,True
4,1750,0001104659-24-037408,AAR CORP,20240229,10-Q,Q3,3720.0,20240321,3.0,2.021800e+09,...,2.216533e+09,4.960000e+07,NaN,2.546667e+07,2024-03-21,NaT,NaN,0.0,False,True
